In [0]:
-- 1:1 comparison: SAME guests, official gold RFM segment (c360_f_rfm_model) vs 4Y LBE RFM segment
WITH
params AS (
  SELECT DATE'2026-10-01' AS as_of, 48 AS lookback_months
),
bounds AS (
  SELECT
    as_of,
    CAST(date_format(add_months(as_of, -lookback_months), 'yyyyMMdd') AS INT)   AS min_id,
    CAST(date_format(as_of, 'yyyyMMdd') AS INT)                                 AS max_id
  FROM params
),
seg_order AS (
  SELECT * FROM VALUES
    (1, 'Champions'), (2, 'Potential Loyalist'), (3, 'Value Conscious Loyals'), (4, 'Promising'),
    (5, 'New'), (6, 'At Risk'), (7, 'Fleeting'), (8, 'High Value Lapsed'),
    (9, 'Not in gold RFM (no theatre visit in 12M)')
  AS t(ord, segment)
),

-- ---------------- official gold RFM: latest snapshot before as_of ----------------
rfm_snap AS (
  SELECT max(CAST(m.SnapshotDateId AS INT)) AS snap_id
  FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model m
  CROSS JOIN bounds b
  WHERE CAST(m.SnapshotDateId AS INT) < b.max_id
),
gold_rfm AS (
  SELECT m.CDE_ID, max(m.Segment) AS gold_segment
  FROM cpx_dataanalytics_gold.customer360.c360_f_rfm_model m
  JOIN rfm_snap s ON CAST(m.SnapshotDateId AS INT) = s.snap_id
  GROUP BY m.CDE_ID
),

-- ---------------- 4Y LBE visits (same LBE visit rule as the feature table) ----------------
lbe_locs AS (
  SELECT LocationID
  FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  GROUP BY LocationID
  HAVING max(CASE WHEN LC_Trade_Brand_Description IN ('Rec Room', 'Playdium') THEN 1 ELSE 0 END) = 1
),
lines AS (
  SELECT
    t.CDE_ID, t.VisitDateId, t.LocationId, t.Net_Revenue,
    CASE
      WHEN i.Item_Class_Description1 IN (
        'TRR - AMUSEMENT - GAMES', 'TRR - AMUSEMENT - PHYSICAL GAMES', 'TRR - PROMO - AMUSEMENT'
      ) THEN 'games'
      WHEN i.Item_Class_Description1 IN (
        'PDM - MENU FOOD - CANDY',
        'TRR - INV ALC BEV - BOTTLE', 'TRR - INV ALC BEV - CAN', 'TRR - INV ALC BEV - WINE',
        'TRR - INV ALC BEV - LIQUOR', 'TRR - INV BEV - NA BEV',
        'TRR - MENU ALC BEV - COCKTAIL', 'TRR - MENU ALC BEV - DRAUGHT BEER',
        'TRR - MENU ALC BEV - LIQUOR', 'TRR - MENU ALC BEV - WINE',
        'TRR - MENU BEV - NA COLD BEV', 'TRR - MENU BEV - NA HOT BEV',
        'TRR - MENU FOOD - APP', 'TRR - MENU FOOD - DESSERT', 'TRR - MENU FOOD - KIDS',
        'TRR - MENU FOOD - PIZZA', 'TRR - MENU FOOD - POUTINE', 'TRR - MENU FOOD - SANDWICHES',
        'TRR - MENU FOOD - ENTRÉE', 'TRR - MENU FOOD - SALAD', 'TRR - MENU FOOD - BRUNCH',
        'TRR - MOD MENU - ADD-ON',
        'VHO-LICENSED FOOD', 'VHO-LICENSED DRINKS', 'VHO-CANDY', 'VHO-DRINK', 'VIP Food - Addons'
      ) THEN 'food'
    END AS lbe_bucket
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN lbe_locs l ON t.LocationId = l.LocationID
  LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON t.ItemSkey = i.itemskey
  CROSS JOIN bounds b
  WHERE t.CDE_ID IS NOT NULL
    AND t.VisitDateId >= b.min_id
    AND t.VisitDateId <  b.max_id
),
visits AS (
  SELECT
    CDE_ID,
    to_date(CAST(VisitDateId AS STRING), 'yyyyMMdd')                            AS visit_date,
    sum(CASE WHEN lbe_bucket IN ('food', 'games') THEN Net_Revenue ELSE 0 END)  AS spend
  FROM lines
  GROUP BY CDE_ID, VisitDateId, LocationId
  HAVING sum(CASE WHEN lbe_bucket IN ('food', 'games') THEN Net_Revenue ELSE 0 END) > 0
),
rfm AS (
  SELECT
    v.CDE_ID,
    datediff(b.as_of, max(v.visit_date))                          AS recency_4y,
    count(*)                                                      AS freq_4y,
    sum(v.spend) / count(*)                                       AS monetary_4y,
    count_if(v.visit_date >= add_months(b.as_of, -12))            AS freq_p12m
  FROM visits v
  CROSS JOIN bounds b
  GROUP BY v.CDE_ID, b.as_of
),

-- 4Y cut-offs on ALL 4Y LBE guests (the proposed model)
cut AS (
  SELECT
    percentile_approx(recency_4y,  array(0.25, 0.5, 0.75))                                  AS r_cut,
    CAST(percentile_approx(CASE WHEN freq_4y >= 2 THEN freq_4y END, 0.333) AS INT)          AS f33,
    CAST(greatest(percentile_approx(CASE WHEN freq_4y >= 2 THEN freq_4y END, 0.667),
                  percentile_approx(CASE WHEN freq_4y >= 2 THEN freq_4y END, 0.333) + 1) AS INT) AS f67,
    percentile_approx(monetary_4y, array(0.25, 0.5, 0.75))                                  AS m_cut
  FROM rfm
),
scored AS (
  SELECT
    r.CDE_ID,
    CASE WHEN r.recency_4y <= c.r_cut[0] THEN 4
         WHEN r.recency_4y <= c.r_cut[1] THEN 3
         WHEN r.recency_4y <= c.r_cut[2] THEN 2
         ELSE 1 END                                  AS r,
    CASE WHEN r.freq_4y = 1 THEN 1
         WHEN r.freq_4y <= c.f33 THEN 2
         WHEN r.freq_4y <= c.f67 THEN 3
         ELSE 4 END                                  AS f_s,
    CASE WHEN r.monetary_4y <= c.m_cut[0] THEN 1
         WHEN r.monetary_4y <= c.m_cut[1] THEN 2
         WHEN r.monetary_4y <= c.m_cut[2] THEN 3
         ELSE 4 END                                  AS m
  FROM rfm r
  CROSS JOIN cut c
  WHERE r.freq_p12m > 0                              -- SAME population: LBE visit in P12M (change to r.freq_4y > 0 for all 4Y guests)
),
pop AS (
  SELECT
    coalesce(g.gold_segment, 'Not in gold RFM (no theatre visit in 12M)') AS gold_segment,
    CASE WHEN s.f_s = 1 AND s.r = 1 AND s.m >= 3     THEN 'High Value Lapsed'
         WHEN s.f_s = 1 AND s.r <= 2                 THEN 'Fleeting'
         WHEN s.f_s = 1                              THEN 'New'
         WHEN s.r = 1                                THEN 'At Risk'
         WHEN s.f_s = 2                              THEN 'Promising'
         WHEN s.m = 1                                THEN 'Value Conscious Loyals'
         WHEN s.r = 4 AND s.f_s = 4 AND s.m >= 3     THEN 'Champions'
         ELSE 'Potential Loyalist' END                AS lbe_4y_segment
  FROM scored s
  LEFT JOIN gold_rfm g ON g.CDE_ID = s.CDE_ID
),
e AS (SELECT gold_segment   AS segment, count(*) AS n FROM pop GROUP BY 1),
l AS (SELECT lbe_4y_segment AS segment, count(*) AS n FROM pop GROUP BY 1)

SELECT
  o.segment,
  coalesce(e.n, 0)                                                   AS gold_rfm_guests,
  round(100 * coalesce(e.n, 0) / sum(coalesce(e.n, 0)) OVER (), 1)   AS gold_rfm_pct,
  coalesce(l.n, 0)                                                   AS lbe_4y_guests,
  round(100 * coalesce(l.n, 0) / sum(coalesce(l.n, 0)) OVER (), 1)   AS lbe_4y_pct,
  round(100 * coalesce(l.n, 0) / sum(coalesce(l.n, 0)) OVER (), 1)
    - round(100 * coalesce(e.n, 0) / sum(coalesce(e.n, 0)) OVER (), 1) AS change_pts,
  (SELECT snap_id FROM rfm_snap)                                     AS gold_snapshot_used
FROM seg_order o
LEFT JOIN e ON e.segment = o.segment
LEFT JOIN l ON l.segment = o.segment
ORDER BY o.ord;